В этом ноутбуке готовим второй вариант данных для MLP с categorical embeddings

In [1]:
import pandas as pd
import numpy as np

import os


X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv")["target_log_price"]
y_val = pd.read_csv("y_val.csv")["target_log_price"]
y_test = pd.read_csv("y_test.csv")["target_log_price"]

train_df = pd.read_csv("train_df.csv")
val_df = pd.read_csv("val_df.csv")
test_df = pd.read_csv("test_df.csv")

final_dataset = pd.read_csv("final_dataset_before_split.csv")
"""

X_train = pd.read_csv("outputs/X_train.csv")
X_val = pd.read_csv("outputs/X_val.csv")
X_test = pd.read_csv("outputs/X_test.csv")

y_train = pd.read_csv("outputs/y_train.csv")["target_log_price"]
y_val = pd.read_csv("outputs/y_val.csv")["target_log_price"]
y_test = pd.read_csv("outputs/y_test.csv")["target_log_price"]

train_df = pd.read_csv("outputs/train_df.csv")
val_df = pd.read_csv("outputs/val_df.csv")
test_df = pd.read_csv("outputs/test_df.csv")

final_dataset = pd.read_csv("outputs/final_dataset_before_split.csv")
"""

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (3698, 45)
X_val: (793, 45)
X_test: (793, 45)
y_train: (3698,)
y_val: (793,)
y_test: (793,)


In [2]:
X_train = X_train.drop(columns=["brand_model"])
X_val = X_val.drop(columns=["brand_model"])
X_test = X_test.drop(columns=["brand_model"])

In [3]:
models = X_train["model"].value_counts()
m_ok = models[models >= 10].index

X_train["m_other"] = "Other"
X_val["m_other"] = "Other"
X_test["m_other"] = "Other"

X_train.loc[X_train["model"].isin(m_ok), "m_other"] = X_train["model"]
X_val.loc[X_val["model"].isin(m_ok), "m_other"] = X_val["model"]
X_test.loc[X_test["model"].isin(m_ok), "m_other"] = X_test["model"]

In [4]:
cities = X_train["city_name"].value_counts()
cc_ok = cities[cities >= 10].index

X_train["cc_other"] = "Other"
X_val["cc_other"] = "Other"
X_test["cc_other"] = "Other"

X_train.loc[X_train["city_name"].isin(cc_ok), "cc_other"] = X_train["city_name"]
X_val.loc[X_val["city_name"].isin(cc_ok), "cc_other"] = X_val["city_name"]
X_test.loc[X_test["city_name"].isin(cc_ok), "cc_other"] = X_test["city_name"]

In [5]:
colors = X_train["base_color"].value_counts()
c_ok = colors[colors >= 10].index

X_train["c_other"] = "Other"
X_val["c_other"] = "Other"
X_test["c_other"] = "Other"

X_train.loc[X_train["base_color"].isin(c_ok), "c_other"] = X_train["base_color"]
X_val.loc[X_val["base_color"].isin(c_ok), "c_other"] = X_val["base_color"]
X_test.loc[X_test["base_color"].isin(c_ok), "c_other"] = X_test["base_color"]

In [6]:
bodies = X_train["body_type"].value_counts()
b_ok = bodies[bodies >= 10].index

X_train["b_other"] = "Other"
X_val["b_other"] = "Other"
X_test["b_other"] = "Other"

X_train.loc[X_train["body_type"].isin(b_ok), "b_other"] = X_train["body_type"]
X_val.loc[X_val["body_type"].isin(b_ok), "b_other"] = X_val["body_type"]
X_test.loc[X_test["body_type"].isin(b_ok), "b_other"] = X_test["body_type"]

In [7]:
print(X_train["model"].nunique(), X_train["m_other"].nunique())
print( X_train["city_name"].nunique(), X_train["cc_other"].nunique())
print(X_train["base_color"].nunique(), X_train["c_other"].nunique())
print( X_train["body_type"].nunique(), X_train["b_other"].nunique())

346 84
94 21
21 16
14 12


In [8]:
other_col = ["m_other", "cc_other", "c_other", "b_other"]

for i in other_col:
    print(i, "train", X_train[i].nunique(), "val", X_val[i].nunique(), "test", X_test[i].nunique())

m_other train 84 val 83 test 82
cc_other train 21 val 21 test 20
c_other train 16 val 16 test 16
b_other train 12 val 12 test 12


Списки признаков для embedding

Разделяем признаки на три группы

num_col - числовые признаки, которые масштабируем через StandardScaler

bin_col - бинарные признаки 0/1, которые добавляем к числовым без кодирования

col_emb - категориальные признаки, которые будем кодировать в целочисленные индексы для embedding

In [9]:
col_emb = [
    "brand",
    "m_other",
    "b_other",
    "body_segment",
    "fuel_type",
    "transmission",
    "drive_type",
    "cc_other",
    "region",
    "city_tier",
    "age_bucket",
    "c_other"
]

num_col = [
    "year",
    "engine_volume",
    "mileage",
    "imgs_count",
    "g_start_year",
    "g_end_year",
    "g_number",
    "g_age_car_release",
    "g_span",
    "car_age",
    "log_mileage",
    "mileage_per_year",
    "log_mileage_per_year",
    "log_imgs_count"
]

bin_col = [
    "steering_wheel",
    "kz_registration",
    "g_is_current",
    "g_is_restyling",
    "generation_number_missing",
    "is_premium",
    "is_big_city",
    "is_electric",
    "is_new_car",
    "is_recent_car",
    "is_old_car",
    "is_metallic",
    "color_known",
    "has_low_mileage",
    "has_high_mileage",
    "is_automatic_like",
    "has_few_photos",
    "has_many_photos"
]

In [10]:
all_cols = col_emb + num_col + bin_col

print(len(all_cols))

44


Всего признаков до embedding-кодирования: 44

Масштабирование числовых признаков

Бинарные признаки уже представлены в формате 0/1, поэтому их не нужно масштабировать. Объединим масштабированные числовые признаки и бинарные признаки. Получаем numeric-блок

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train[num_col])
X_val_num_scaled = scaler.transform(X_val[num_col])
X_test_num_scaled = scaler.transform(X_test[num_col])

In [12]:
X_train_bin = X_train[bin_col].values
X_val_bin = X_val[bin_col].values
X_test_bin = X_test[bin_col].values

X_train_num = np.hstack([X_train_num_scaled, X_train_bin])
X_val_num = np.hstack([X_val_num_scaled, X_val_bin])
X_test_num = np.hstack([X_test_num_scaled, X_test_bin])

print(X_train_num.shape)
print(X_val_num.shape)
print(X_test_num.shape)

(3698, 32)
(793, 32)
(793, 32)


Кодирование категориальных признаков в индексы

Индекс 0 резервируем для неизвестных категорий

Словари категорий строим только на X_train, а затем применяем к X_val и X_test

In [13]:
cat_mappings = {}

for i in col_emb:
    values = sorted(X_train[i].unique())

    mapping = {}
    index = 1

    for j in values:
        mapping[j] = index
        index = index + 1

    cat_mappings[i] = mapping

In [14]:
X_train_cat = pd.DataFrame(index=X_train.index)
X_val_cat = pd.DataFrame(index=X_val.index)
X_test_cat = pd.DataFrame(index=X_test.index)

for i in col_emb:
    mapping = cat_mappings[i]

    X_train_cat[i] = 0
    X_val_cat[i] = 0
    X_test_cat[i] = 0

    for category in mapping:
        index = mapping[category]

        X_train_cat.loc[X_train[i] == category, i] = index
        X_val_cat.loc[X_val[i] == category, i] = index
        X_test_cat.loc[X_test[i] == category, i] = index

In [15]:
print(X_train_cat.shape)
print(X_val_cat.shape)
print(X_test_cat.shape)

X_train_cat.head()

(3698, 12)
(793, 12)
(793, 12)


,brand,m_other,b_other,body_segment,fuel_type,transmission,drive_type,cc_other,region,city_tier,age_bucket,c_other
0,6,67,3,5,2,1,3,21,16,4,3,13
1,6,41,9,3,2,1,3,5,2,3,3,12
2,6,41,9,3,2,1,1,4,4,3,3,12
3,8,53,2,5,6,1,3,4,4,3,6,4
4,7,60,9,3,2,4,2,4,4,3,6,4


In [16]:
cat_sizes = {}

for i in col_emb:
    cat_sizes[i] = len(cat_mappings[i]) + 1

cat_sizes

{'brand': 9,
 'm_other': 85,
 'b_other': 13,
 'body_segment': 7,
 'fuel_type': 7,
 'transmission': 5,
 'drive_type': 4,
 'cc_other': 22,
 'region': 18,
 'city_tier': 5,
 'age_bucket': 7,
 'c_other': 17}

In [17]:
for i in col_emb:
    print(i, cat_sizes[i], "max train:", X_train_cat[i].max(), "max val:", X_val_cat[i].max(), "max test:", X_test_cat[i].max())

brand 9 max train: 8 max val: 8 max test: 8
m_other 85 max train: 84 max val: 84 max test: 84
b_other 13 max train: 12 max val: 12 max test: 12
body_segment 7 max train: 6 max val: 6 max test: 6
fuel_type 7 max train: 6 max val: 6 max test: 6
transmission 5 max train: 4 max val: 4 max test: 4
drive_type 4 max train: 3 max val: 3 max test: 3
cc_other 22 max train: 21 max val: 21 max test: 21
region 18 max train: 17 max val: 17 max test: 17
city_tier 5 max train: 4 max val: 4 max test: 4
age_bucket 7 max train: 6 max val: 6 max test: 6
c_other 17 max train: 16 max val: 16 max test: 16


In [18]:
OUTPUT_DIR = "mlp_data_embeddings"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(X_train_num, columns=num_col + bin_col).to_csv(f"{OUTPUT_DIR}/X_train_num.csv", index=False)
pd.DataFrame(X_val_num, columns=num_col + bin_col).to_csv(f"{OUTPUT_DIR}/X_val_num.csv", index=False)
pd.DataFrame(X_test_num, columns=num_col + bin_col).to_csv(f"{OUTPUT_DIR}/X_test_num.csv", index=False)

X_train_cat.to_csv(f"{OUTPUT_DIR}/X_train_cat.csv", index=False)
X_val_cat.to_csv(f"{OUTPUT_DIR}/X_val_cat.csv", index=False)
X_test_cat.to_csv(f"{OUTPUT_DIR}/X_test_cat.csv", index=False)

y_train.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_val.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_val.csv", index=False)
y_test.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)